### **Create a secret scope**
https://adb-id.10.azuredatabricks.net/#secrets/createScope

**End-to-End Task: ADLS → Databricks → Silver → Gold**

🧩 Scenario

Raw customer order data lands in ADLS (Bronze). You need to clean, transform, and build analytics-ready tables.

**🥉 Step 1: Bronze Layer (Raw Data in ADLS)**

In [0]:
secrets_list = dbutils.secrets.list("adb-secret-scope")
display(secrets_list)

In [0]:
key = dbutils.secrets.get(scope="adb-secret-scope", key="gen2key")

spark.conf.set(
  "fs.azure.account.key.stgdataengaccountctlus.dfs.core.windows.net",
  key
)

In [0]:
key = dbutils.secrets.get(scope="adb-secret-scope", key="gen2key")

spark.conf.set(
  "fs.azure.account.key.stgdataengaccountctlus.dfs.core.windows.net",
  key
)

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfs://bronze@stgdataengaccountctlus.dfs.core.windows.net/*.csv")

df.show()

Step 2: Silver Layer (Cleaned Data)
- Transformations Required
- Remove duplicates
- Handle nulls (customer_name)
- Convert data types
- Add new column: total_amount = price * quantity
- Standardize column names

In [0]:
from pyspark.sql.functions import col, when

df_clean = df.dropDuplicates()

df_clean = df_clean.withColumn(
    "customer_name",
    when(col("customer_name").isNull(), "Unknown").otherwise(col("customer_name"))
)

df_clean = df_clean.withColumn("price", col("price").cast("double")) \
                   .withColumn("quantity", col("quantity").cast("int"))

df_clean = df_clean.withColumn("total_amount", col("price") * col("quantity"))

In [0]:
df_clean.display()

**💾 Save to Silver Layer**


In [0]:
df_clean.write.format("parquet") \
    .mode("overwrite") \
    .save("abfs://silver@stgdataengaccountctlus.dfs.core.windows.net/orders")


Step 3: Gold Layer (Business Aggregations)

🎯 Business Use Case

Create insights for dashboards:
- Total revenue per category
- Total orders per customer
- Top customers

In [0]:
from pyspark.sql.functions import sum, count

# Revenue by category
df_gold_category = df_clean.groupBy("category") \
    .agg(sum("total_amount").alias("total_revenue"))

# Orders per customer
df_gold_customer = df_clean.groupBy("customer_id", "customer_name") \
    .agg(count("order_id").alias("total_orders"),
         sum("total_amount").alias("total_spent"))

**💾 Save Gold Layer**

In [0]:
df_gold_category.write.format("parquet") \
    .mode("overwrite") \
    .save("abfs://gold@stgdataengaccountctlus.dfs.core.windows.net/category_revenue")

df_gold_customer.write.format("parquet") \
    .mode("overwrite") \
    .save("abfs://gold@stgdataengaccountctlus.dfs.core.windows.net/customer_metrics")